# DiffuScene Setup on Google Colab
Notebook này thiết lập môi trường và tải sẵn các pretrained model, dataset đã được xử lý (preprocessed) để không cần phải chạy lại các bước tốn thời gian.

In [2]:
# Clone source code (Bỏ comment phần này nếu bạn upload notebook lên một Colab trống)
!git clone https://github.com/AdamHermes/DiffuScene.git
%cd DiffuScene


Cloning into 'DiffuScene'...
remote: Enumerating objects: 461, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 461 (delta 153), reused 118 (delta 115), pack-reused 251 (from 2)
Receiving objects: 100% (461/461), 1.63 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (268/268), done.
/content/DiffuScene


In [3]:
# Cài đặt các thư viện cần thiết trực tiếp trên Colab
!pip install simple_3dviz pytorch-fast-transformers num2words einops_exts kornia rotary_embedding_torch transformers pyvista wandb trimesh pyrr
!pip install git+https://github.com/openai/CLIP.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.6/93.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.0/234.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 7.5 MB/s eta 0:00:00
   ━━

In [4]:
%%bash

# Build các module mở rộng
python setup.py build_ext --inplace
pip install -e .

# Cài đặt ChamferDistancePytorch
cd ChamferDistancePytorch/chamfer3D
python setup.py install

<_io.TextIOWrapper name='scene_synthesis/__init__.py' mode='r' encoding='utf-8'>
meta {'description': ''}
running build_ext
Obtaining file:///content/DiffuScene
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for scene-synthesis
running install
running bdist_egg
running egg_info
creating chamfer_3D.egg-info
writing chamfer_3D.egg-info/PKG-INFO
writing dependency_links to chamfer_3D.egg-info/dependency_links.txt
writing top-level names to chamfer_3D.egg-info/top_level.txt
writing manifest file 'chamfer_3D.egg-info/SOURCES.txt'
reading manifest file 'chamfer_3D.egg-info/SOURCES.txt'
writing manifest file 'chamfer_3D.egg-info/SOURCES.txt'
installing library code to build/bdist.linux-x86_64/egg
running install_lib
running build_ext
building 'chamfer_3D' extension
creating build/temp.linux-x86_64-cpython-312/content/DiffuScene/ChamferDistancePytorch/chamfer3D
/usr/local/cuda/bin/nvcc -I/usr/local/lib/python3.12

/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'extra_cuda_cflags'
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!zipinfo -1 /content/drive/MyDrive/DiffuScene/3D-FUTURE-model-processed.zip | wc -l

## Tải Pretrained Models và Preprocessed Datasets
Sử dụng `gdown` để tải trực tiếp từ Google Drive của tác giả.

In [5]:
!pip install --upgrade gdown

import os
os.makedirs("models", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

# 1. Pretrained models of DiffuScene
#!gdown 1pk9AzGcBz_kRfmRzvFNDW5byk4MwbXEm -O models/pretrained_models.zip


# 3. Preprocessed 3D-Front dataset
#!gdown 1UNSFN0kULyOzUErDPVvkKYbmfzA-4MsG -O datasets/3D-Front_preprocessed.zip


# Giải nén các file (Có thể mất một lúc tùy vào kích thước)
!unzip -q /content/drive/MyDrive/DiffuScene/pretrained_diffusion.zip -d models/
!unzip -q /content/drive/MyDrive/DiffuScene/3d_front_processed.zip -d datasets/


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/chamfer_3D-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


## Chạy Evaluation
Môi trường và dữ liệu đã sẵn sàng. Bạn có thể sử dụng `%%bash` cell như dưới đây để thực hiện eval/generate. (Cần kiểm tra lại các script shell `generate.sh` để trỏ đúng đường dẫn dataset/models bạn vừa giải nén nếu mặc định không khớp).

In [ ]:
%%bash

# Chạy ví dụ generate (Mở uncomment để chạy)
# ./run/generate.sh
# ./run/generate_text.sh

In [1]:
!pip install open3d

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/chamfer_3D-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [2]:
import nltk

nltk.download("cmudict")

[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


True

In [6]:
%cd /content/DiffuScene/scripts

/content/DiffuScene/scripts


In [4]:
!ls "/content/drive/MyDrive/DiffuScene"

3d_front_processed.zip	       pretrained_diffusion.zip  run18_2  UncondGen
3D-FUTURE-model		       run0_3			 run3_5   UncondGen.zip
3D-FUTURE-model-processed.zip  run13_5			 run8_5


In [4]:
!apt-get install -y xvfb
!pip install pyvirtualdisplay

from pyvirtualdisplay import Display

display = Display(visible=0, size=(512, 512))
display.start()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/chamfer_3D-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [7]:
exp_dir = "../models"
exp_name = "pretrained_diffusion/diningrooms_uncond"

config = "../config/uncond/diffusion_diningrooms_instancond_lat32_v.yaml"
threed_future = "../datasets/3d_front_processed/threed_future_model_diningroom.pkl"

weight_file = f"{exp_dir}/{exp_name}/model_82000"
output_dir = f"{exp_dir}/{exp_name}/gen_top2down_notexture_nofloor"

In [ ]:
!python -u generate_diffusion.py {config} {output_dir} {threed_future} \
    --weight_file {weight_file} \
    --n_sequences 5 \
    --fix_order \
    --start_index 2 \
    --render_top2down \
    --save_mesh \
    --no_texture \
    --without_floor \
    --clip_denoised \
    --retrive_objfeats

Running code on cuda:0
NO PERM AUG in test
encoding type : cached_diffusion_cosin_angle_objfeatsnorm_lat32_wocm_no_prm
Applying threed_front_diningroom filtering
bounds_objfeats of dataset: [0.9673417806625366, -4.929184436798096, 6.195308208465576]
bounds_objfeats_32 of dataset: [0.9735163450241089, -5.804656028747559, 5.496402740478516]
rendered_scene is : rendered_scene_256.png
use lat32 as objfeats
use consin_angles instead of original angles, AND use normalized objfeats
permute keys are: ['class_labels', 'translations', 'sizes', 'angles', 'objfeats_32']
Start filtering objects from /content/drive/MyDrive/DiffuScene/3D-FUTURE-model ...
Loaded 1980 3D-FUTURE models
Applying threed_front_diningroom filtering
bounds_objfeats of dataset: [0.9673417806625366, -4.929184436798096, 6.195308208465576]
bounds_objfeats_32 of dataset: [0.9735163450241089, -5.804656028747559, 5.496402740478516]
rendered_scene is : rendered_scene_256.png
use lat32 as objfeats
use consin_angles instead of origina

In [ ]:
!cp -r "/content/DiffuScene/models/pretrained_diffusion/diningrooms_uncond/gen_top2down_notexture_nofloor" "/content/drive/MyDrive/DiffuScene/run0_5"

In [ ]:
!git pull origin master

From https://github.com/AdamHermes/DiffuScene
 * branch            master     -> FETCH_HEAD
Already up to date.


In [ ]:
!git checkout cf0b74517cf00d6834d57692a31c46278bdeef58

HEAD is now at cf0b745 check list dir
